In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime


df = pd.read_excel("/content/flickr zong.xlsx")

df['datetaken'] = pd.to_datetime(df['date_taken'], errors='coerce')

df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')

df.dropna(subset=['latitude', 'longitude'], inplace=True)

CENTER_LAT = 51.5246
CENTER_LNG = -0.1340

DELTA_LAT = 0.06
DELTA_LNG = 0.09

LAT_MIN = CENTER_LAT - DELTA_LAT
LAT_MAX = CENTER_LAT + DELTA_LAT
LNG_MIN = CENTER_LNG - DELTA_LNG
LNG_MAX = CENTER_LNG + DELTA_LNG

def in_city(lat, lng):
    return (LAT_MIN <= lat <= LAT_MAX) and (LNG_MIN <= lng <= LNG_MAX)

df['in_city'] = df.apply(lambda r: in_city(r['latitude'], r['longitude']), axis=1)

def classify_user(user_df):
    total_photos = len(user_df)
    city_photos = user_df['in_city'].sum()
    ratio = city_photos / total_photos

    dates = user_df['datetaken'].dropna()
    if len(dates) > 1:
        day_span = (dates.max() - dates.min()).days
    else:
        day_span = 0

    if ratio >= 0.50 and day_span >= 90:
        cls = "RESIDENT"
    else:
        cls = "TOURIST"

    return pd.Series({
        "total_photos": total_photos,
        "city_photos": city_photos,
        "city_ratio": ratio,
        "day_span": day_span,
        "classification": cls
    })

result = df.groupby("owner").apply(classify_user, include_groups=False).reset_index()

result.to_csv("/content/flickr_user_classification_London.csv", index=False)
print("flickr_user_classification_London.csv")

完成！输出文件：flickr_user_classification_London.csv


In [ ]:
import pandas as pd
import folium
from datetime import datetime
from google.colab import drive

CENTER_LAT = 51.5246
CENTER_LNG = -0.1340

DELTA_LAT = 0.06
DELTA_LNG = 0.09

LAT_MIN = CENTER_LAT - DELTA_LAT
LAT_MAX = CENTER_LAT + DELTA_LAT
LNG_MIN = CENTER_LNG - DELTA_LNG
LNG_MAX = CENTER_LNG + DELTA_LNG

def in_city(lat, lng):
    return (LAT_MIN <= lat <= LAT_MAX) and (LNG_MIN <= lng <= LNG_MAX)

def classify_user(user_df):
    total_photos = len(user_df)
    city_photos = user_df['in_city'].sum()
    ratio = city_photos / total_photos

    dates = user_df['datetaken'].dropna()
    if len(dates) > 1:
        day_span = (dates.max() - dates.min()).days
    else:
        day_span = 0

    if ratio >= 0.50 and day_span >= 90:
        cls = "RESIDENT"
    else:
        cls = "TOURIST"

    return pd.Series({"classification": cls})

original_photos_df = pd.read_excel(
    "/content/flickr总.xlsx"
)

original_photos_df['datetaken'] = pd.to_datetime(
    original_photos_df['date_taken'],
    errors='coerce'
)

original_photos_df['latitude'] = pd.to_numeric(original_photos_df['latitude'], errors='coerce')
original_photos_df['longitude'] = pd.to_numeric(original_photos_df['longitude'], errors='coerce')
original_photos_df.dropna(subset=['latitude', 'longitude'], inplace=True)

original_photos_df['in_city'] = (
    (original_photos_df['latitude'] >= LAT_MIN) &
    (original_photos_df['latitude'] <= LAT_MAX) &
    (original_photos_df['longitude'] >= LNG_MIN) &
    (original_photos_df['longitude'] <= LNG_MAX)
)

user_classification_df = (
    original_photos_df
    .groupby("owner")
    .apply(classify_user, include_groups=False)
    .reset_index()
)

df = original_photos_df.merge(
    user_classification_df[['owner', 'classification']],
    on='owner',
    how='left'
)

df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df.dropna(subset=['latitude', 'longitude', 'classification'], inplace=True)

m = folium.Map(
    location=[CENTER_LAT, CENTER_LNG],
    tiles="Cartodb dark_matter",
    zoom_start=13,
    control_scale=True
)

folium.Rectangle(
    bounds=[[LAT_MAX, LNG_MIN], [LAT_MIN, LNG_MAX]],
    fill=True,
    fill_opacity=0.1,
    weight=1,
    color="cornflowerblue",
).add_to(m)

color_map = {
    "RESIDENT": "blue",
    "TOURIST": "red"
}

for _, row in df.iterrows():
    if row['classification'] in color_map:
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=3,
            color=color_map[row["classification"]],
            fill=True,
            fill_opacity=0.6
        ).add_to(m)

m.save("london_resident_vs_tourist_map.html")
print("完成！地图文件：london_resident_vs_tourist_map.html")

完成！地图文件：london_resident_vs_tourist_map.html


In [ ]:
import pandas as pd
import folium
import json

CENTER_LAT = 51.5246
CENTER_LNG = -0.1340

DELTA_LAT = 0.06
DELTA_LNG = 0.09

LAT_MIN = CENTER_LAT - DELTA_LAT
LAT_MAX = CENTER_LAT + DELTA_LAT
LNG_MIN = CENTER_LNG - DELTA_LNG
LNG_MAX = CENTER_LNG + DELTA_LNG

df = pd.read_excel(
    "/content/flickr zong.xlsx"
)

df["datetaken"] = pd.to_datetime(df["date_taken"], errors="coerce")

df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df.dropna(subset=['latitude', 'longitude'], inplace=True)

df["in_city"] = (
    (df['latitude'] >= LAT_MIN) &
    (df['latitude'] <= LAT_MAX) &
    (df['longitude'] >= LNG_MIN) &
    (df['longitude'] <= LNG_MAX)
)

def classify_user(user_df):
    total_photos = len(user_df)
    city_photos = user_df["in_city"].sum()
    ratio = city_photos / total_photos

    dates = user_df["datetaken"].dropna()
    day_span = (dates.max() - dates.min()).days if len(dates) > 1 else 0

    if ratio >= 0.50 and day_span >= 90:
        return "RESIDENT"
    else:
        return "TOURIST"

user_class = (
    df.groupby("owner")
    .apply(classify_user, include_groups=False)
    .reset_index()
)
user_class.columns = ["owner", "classification"]

df = df.merge(user_class, on="owner", how="left")

residents = df[df["classification"] == "RESIDENT"]
tourists = df[df["classification"] == "TOURIST"]

residents.to_csv("/content/resident_points_london.csv", index=False)
tourists.to_csv("/content/tourist_points_london.csv", index=False)

print("CSV")

def df_to_geojson(subdf):
    geojson = {
        "type": "FeatureCollection",
        "features": []
    }

    for _, row in subdf.iterrows():
        geojson["features"].append({
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row["longitude"], row["latitude"]]
            },
            "properties": {
                "owner": row["owner"],
                "classification": row["classification"],
                "datetaken": row["datetaken"].isoformat() if pd.notna(row["datetaken"]) else None,
                "title": str(row["title"]) if pd.notna(row.get("title")) else None,
                "description": str(row["description"]) if pd.notna(row.get("description")) else None
            }
        })

    return geojson

resident_geojson = df_to_geojson(residents)
tourist_geojson = df_to_geojson(tourists)

with open('/content/residents_london.geojson', 'w') as f:
    json.dump(resident_geojson, f, indent=2, ensure_ascii=False)

with open('/content/tourists_london.geojson', 'w') as f:
    json.dump(tourist_geojson, f, indent=2, ensure_ascii=False)

print("GeoJSON ")

✅ CSV 已生成
✅ GeoJSON 已生成
